# MRIxFields2026 Task 3 — organizer-pretrained baseline

This notebook is only for the official shared 15-domain Task 3 checkpoint:
stage the released validation inputs into Colab runtime storage, audit them,
run smoke/full inference, build and validate task3.zip, and optionally export
the final archive and JSON reports.

Mounted Drive is never used as a working directory. Code, validation inputs,
and an optional saved checkpoint are copied into /content before use. No
predictions, caches, manifests, or intermediate results are written to Drive.
The final cell is the only opt-in Drive write, and nothing is uploaded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

RUNTIME_ROOT_VALUE = '/content/mrixfields_task3/pretrained'  # @param {type:'string'}
DRIVE_CODE_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields_task3_source'  # @param {type:'string'}
DRIVE_VALIDATION_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields2026/validation_prospective'  # @param {type:'string'}
DRIVE_CHECKPOINT_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields_weights/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'  # @param {type:'string'}
DRIVE_EXPORT_ROOT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/pretrained'  # @param {type:'string'}

RUNTIME_ROOT = Path(RUNTIME_ROOT_VALUE)
DRIVE_CODE_SOURCE = Path(DRIVE_CODE_SOURCE_VALUE)
DRIVE_VALIDATION_SOURCE = Path(DRIVE_VALIDATION_SOURCE_VALUE)
DRIVE_CHECKPOINT_SOURCE = Path(DRIVE_CHECKPOINT_SOURCE_VALUE)
DRIVE_EXPORT_ROOT = Path(DRIVE_EXPORT_ROOT_VALUE)
CODE_DIR = RUNTIME_ROOT / 'code'
VALIDATION_DIR = RUNTIME_ROOT / 'data' / 'validation_prospective'
WEIGHTS_DIR = RUNTIME_ROOT / 'weights'
RESULTS_DIR = RUNTIME_ROOT / 'results'
SMOKE_OUTPUT = RESULTS_DIR / 'pretrained_smoke'
FULL_OUTPUT = RESULTS_DIR / 'pretrained_full'
SUBMISSION_DIR = RESULTS_DIR / 'submission_epoch50'
REPORT_DIR = RESULTS_DIR / 'reports'
ARCHIVE = RESULTS_DIR / 'task3.zip'
OFFICIAL_REPO = RUNTIME_ROOT / 'official' / 'MRIxFields2026'
PIPELINE = CODE_DIR / 'task3_pipeline.py'
SUBMISSION_HELPER = CODE_DIR / 'task3_build_submission.py'
REQUIREMENTS = CODE_DIR / 'requirements-colab.txt'

drive_root = Path('/content/drive').resolve()
if RUNTIME_ROOT.resolve() == drive_root or drive_root in RUNTIME_ROOT.resolve().parents:
    raise ValueError('RUNTIME_ROOT must be under /content, not mounted Drive.')
for path in (CODE_DIR, WEIGHTS_DIR, RESULTS_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(RUNTIME_ROOT / '.cache' / 'torch')
os.environ['HF_HOME'] = str(RUNTIME_ROOT / '.cache' / 'huggingface')

for filename in ('task3_pipeline.py', 'task3_build_submission.py', 'requirements-colab.txt'):
    target = CODE_DIR / filename
    source = next(
        (
            candidate for candidate in
            (DRIVE_CODE_SOURCE / filename, Path('/content') / filename)
            if candidate.is_file()
        ),
        None,
    )
    if source is not None:
        shutil.copy2(source, target)
for required in (PIPELINE, SUBMISSION_HELPER, REQUIREMENTS):
    if not required.is_file():
        raise FileNotFoundError(
            f'Missing {required.name}; put it in {DRIVE_CODE_SOURCE} or /content.'
        )
print('Pretrained runtime:', RUNTIME_ROOT)
print('Persistent export (unused until final cell):', DRIVE_EXPORT_ROOT)


## Install the baseline dependencies

Colab supplies CUDA PyTorch. Package caches and downloaded files remain under
the pretrained runtime root.


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)
])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before inference.')
props = torch.cuda.get_device_properties(0)
print(torch.__version__, torch.version.cuda)
print(props.name, round(props.total_memory / 2**30, 1), 'GiB')


## Stage the validation release into runtime

Set the source to the validation split itself. The copy is resumable within the
current Colab session and verifies the relative path/size inventory. After this
cell, all audits and inference read only VALIDATION_DIR under /content.


In [ ]:
def require_runtime_path(path):
    path = Path(path).expanduser().resolve()
    drive_root = Path('/content/drive').resolve()
    if path == drive_root or drive_root in path.parents:
        raise ValueError(f'Working path must be runtime-local, not mounted Drive: {path}')
    return path


def stage_nifti_tree(source, destination, expected_files=None):
    source = Path(source).expanduser().resolve()
    destination = require_runtime_path(destination)
    if not source.is_dir():
        raise FileNotFoundError(f'Data source not found: {source}')
    if source == destination or source in destination.parents or destination in source.parents:
        raise ValueError('Source and runtime destination must be separate trees.')
    source_files = sorted(source.rglob('*.nii.gz'))
    if expected_files is not None and len(source_files) != expected_files:
        raise RuntimeError(
            f'Found {len(source_files)} NIfTI files in {source}; expected {expected_files}.'
        )
    if not source_files:
        raise RuntimeError(f'No NIfTI files found in {source}')
    source_index = {
        path.relative_to(source): path.stat().st_size for path in source_files
    }
    destination.mkdir(parents=True, exist_ok=True)
    for index, (relative, size) in enumerate(source_index.items(), start=1):
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.stat().st_size == size:
            continue
        temporary = target.with_name(f'.{target.name}.part')
        if temporary.exists():
            temporary.unlink()
        shutil.copy2(source / relative, temporary)
        if temporary.stat().st_size != size:
            raise RuntimeError(f'Incomplete runtime copy: {temporary}')
        os.replace(temporary, target)
        if index % 10 == 0 or index == len(source_index):
            print(f'Staged {index}/{len(source_index)} files')
    runtime_index = {
        path.relative_to(destination): path.stat().st_size
        for path in destination.rglob('*.nii.gz')
    }
    if runtime_index != source_index:
        raise RuntimeError('Runtime data inventory differs from the selected source tree.')
    print(f'Runtime data ready: {destination} ({len(runtime_index)} files)')
    return destination


STAGE_VALIDATION_DATA = False  # @param {type:'boolean'}
if STAGE_VALIDATION_DATA:
    stage_nifti_tree(
        DRIVE_VALIDATION_SOURCE, VALIDATION_DIR, expected_files=45
    )
elif not VALIDATION_DIR.is_dir():
    print('Validation staging is OFF. Enable it before audit or inference.')
else:
    print('Reusing runtime validation data:', VALIDATION_DIR)


## Recheck and fetch the pinned official sources

This baseline was prepared against one coherent organizer Git commit and
Hugging Face revision. The cell checks the live default revisions first and
stops if either changed, so code/config/weights cannot be mixed silently.


In [ ]:
OFFICIAL_COMMIT = 'f0f4ab2c54ab878b48ddfea8d13f4da512ca411e'
HF_REPO_ID = 'mrixfields/MRIxFields2026-Baseline'
HF_REVISION = 'f1a7d6b741b64c81871d0856fa2d7295d564c141'
HF_FILE = 'task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'
LIVE_RULES_REVIEWED = False  # @param {type:'boolean'}
LIVE_RULE_URLS = (
    'https://mrixfields.chihucloud.com/2026/tasks.html',
    'https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md',
)

remote_line = subprocess.check_output([
    'git', 'ls-remote', 'https://github.com/MRIxFields/MRIxFields2026.git',
    'refs/heads/main',
], text=True).strip()
live_official_commit = remote_line.split()[0]
if live_official_commit != OFFICIAL_COMMIT:
    raise RuntimeError(
        f'Official main changed to {live_official_commit}; stop and review the '
        f'config, inference code, and submission contract before continuing.'
    )

from huggingface_hub import HfApi
live_hf_revision = HfApi().repo_info(repo_id=HF_REPO_ID).sha
if live_hf_revision != HF_REVISION:
    raise RuntimeError(
        f'Hugging Face main changed to {live_hf_revision}; stop and review '
        'the shared Task 3 checkpoint before continuing.'
    )

if not (OFFICIAL_REPO / '.git').is_dir():
    OFFICIAL_REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        'git', 'clone', 'https://github.com/MRIxFields/MRIxFields2026.git',
        str(OFFICIAL_REPO),
    ])
subprocess.check_call([
    'git', '-C', str(OFFICIAL_REPO), 'checkout', '--detach', OFFICIAL_COMMIT
])
revision = subprocess.check_output(
    ['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert revision == OFFICIAL_COMMIT, revision
live_report = {
    'official_git_revision': revision,
    'huggingface_revision': live_hf_revision,
    'rules_reviewed': LIVE_RULES_REVIEWED,
    'rule_urls': list(LIVE_RULE_URLS),
}
(REPORT_DIR / 'live_source_check.json').write_text(
    json.dumps(live_report, indent=2) + '\n'
)
print(live_report)
if not LIVE_RULES_REVIEWED:
    print('Review both live rule URLs before the final build, then rerun this cell with LIVE_RULES_REVIEWED=True.')


## Stage the official shared checkpoint into runtime

Prefer an existing saved copy when available; otherwise enable the pinned
download. Either route writes the working checkpoint only under /content and
verifies its official byte size and SHA-256.


In [ ]:
USE_SAVED_CHECKPOINT = True  # @param {type:'boolean'}
DOWNLOAD_OFFICIAL_CHECKPOINT = False  # @param {type:'boolean'}
EXPECTED_CHECKPOINT_SIZE = 631346455
EXPECTED_CHECKPOINT_SHA256 = 'c2f8172c1fbdbf0426e22c7d3d2014632f175a38cf818f95e72b1917badabdeb'
CHECKPOINT = WEIGHTS_DIR / HF_FILE
if USE_SAVED_CHECKPOINT and DOWNLOAD_OFFICIAL_CHECKPOINT:
    raise ValueError('Choose the saved checkpoint or the download, not both.')
if USE_SAVED_CHECKPOINT:
    if not DRIVE_CHECKPOINT_SOURCE.is_file():
        raise FileNotFoundError(DRIVE_CHECKPOINT_SOURCE)
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if (
        not CHECKPOINT.is_file()
        or CHECKPOINT.stat().st_size != DRIVE_CHECKPOINT_SOURCE.stat().st_size
    ):
        temporary = CHECKPOINT.with_name('.checkpoint_epoch50.pth.part')
        shutil.copy2(DRIVE_CHECKPOINT_SOURCE, temporary)
        os.replace(temporary, CHECKPOINT)
elif DOWNLOAD_OFFICIAL_CHECKPOINT:
    from huggingface_hub import hf_hub_download
    CHECKPOINT = Path(hf_hub_download(
        repo_id=HF_REPO_ID, filename=HF_FILE, revision=HF_REVISION,
        local_dir=WEIGHTS_DIR,
    ))
elif not CHECKPOINT.is_file():
    raise FileNotFoundError('No runtime checkpoint. Select one staging route.')

digest = hashlib.sha256()
with CHECKPOINT.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
checkpoint_sha256 = digest.hexdigest()
if CHECKPOINT.stat().st_size != EXPECTED_CHECKPOINT_SIZE:
    raise RuntimeError(f'Unexpected checkpoint size: {CHECKPOINT.stat().st_size}')
if checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f'Checkpoint SHA-256 mismatch: {checkpoint_sha256}')
print('Runtime checkpoint:', CHECKPOINT)
print('SHA-256:', checkpoint_sha256)


## Audit and plan

This checks the staged release, NIfTI headers, expected IDs, all three
modalities, and the complete 180-case non-identity plan before a model loads.


In [ ]:
if not VALIDATION_DIR.is_dir():
    raise RuntimeError('Stage validation data into runtime first.')
subprocess.check_call([
    sys.executable, str(PIPELINE), 'audit',
    '--input-root', str(VALIDATION_DIR),
    '--modalities', 'T1W', 'T2W', 'T2FLAIR',
    '--report', str(REPORT_DIR / 'pretrained_input_audit.json'),
])
subprocess.check_call([
    sys.executable, str(PIPELINE), 'plan',
    '--input-root', str(VALIDATION_DIR),
    '--output-dir', str(FULL_OUTPUT),
    '--modalities', 'T1W', 'T2W', 'T2FLAIR',
])


## One full-volume smoke translation

Keep the smoke output separate from the complete baseline result. The verified
joint domain mapping is used, including T2W/T2FLAIR domains 5 through 14.


In [ ]:
RUN_PRETRAINED_SMOKE = False  # @param {type:'boolean'}
if RUN_PRETRAINED_SMOKE:
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'infer',
        '--input-root', str(VALIDATION_DIR),
        '--output-dir', str(SMOKE_OUTPUT),
        '--official-repo', str(OFFICIAL_REPO),
        '--checkpoint', str(CHECKPOINT),
        '--modalities', 'T1W', '--device', 'cuda',
        '--batch-size', '1', '--seed', '20260712',
        '--smoke', '--resume',
    ])
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'validate-outputs',
        '--input-root', str(VALIDATION_DIR),
        '--output-dir', str(SMOKE_OUTPUT),
        '--modalities', 'T1W', '--smoke',
        '--report', str(REPORT_DIR / 'pretrained_smoke_validation.json'),
    ])
else:
    print('Pretrained smoke inference is OFF.')


## Complete organizer-pretrained sweep

Run only after the smoke passes. One checkpoint remains loaded for all 180
predictions. All working outputs stay in the pretrained runtime result tree.


In [ ]:
RUN_PRETRAINED_FULL_SWEEP = False  # @param {type:'boolean'}
if RUN_PRETRAINED_FULL_SWEEP:
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'infer',
        '--input-root', str(VALIDATION_DIR),
        '--output-dir', str(FULL_OUTPUT),
        '--official-repo', str(OFFICIAL_REPO),
        '--checkpoint', str(CHECKPOINT),
        '--modalities', 'T1W', 'T2W', 'T2FLAIR',
        '--device', 'cuda', '--batch-size', '1', '--seed', '20260712',
        '--resume',
    ])
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'validate-outputs',
        '--input-root', str(VALIDATION_DIR),
        '--output-dir', str(FULL_OUTPUT),
        '--modalities', 'T1W', 'T2W', 'T2FLAIR',
        '--report', str(REPORT_DIR / 'pretrained_full_validation.json'),
    ])
else:
    print('Full pretrained sweep is OFF.')


## Build and validate task3.zip

This uses the current pinned official Task 3 builder, validates the exact
task3/ tree and all 180 cropped float32 predictions, and records the archive
SHA-256. It does not upload.


In [ ]:
BUILD_PRETRAINED_TASK3_ZIP = False  # @param {type:'boolean'}
SUBMISSION_REPORT = REPORT_DIR / 'pretrained_submission.json'
if BUILD_PRETRAINED_TASK3_ZIP:
    if not LIVE_RULES_REVIEWED:
        raise RuntimeError(
            'Review the live competition and submission rules before final build.'
        )
    if SUBMISSION_DIR.exists() or ARCHIVE.exists():
        raise RuntimeError(
            'Use a fresh runtime result path; refusing to mix stale submission files.'
        )
    subprocess.check_call([
        sys.executable, str(SUBMISSION_HELPER),
        '--official-repo', str(OFFICIAL_REPO),
        '--predictions-dir', str(FULL_OUTPUT),
        '--validation-input-root', str(VALIDATION_DIR),
        '--output-dir', str(SUBMISSION_DIR),
        '--archive', str(ARCHIVE),
        '--report', str(SUBMISSION_REPORT),
        '--sweep-mode', 'pro_pretrained',
        '--sweep-epoch', 'epoch50',
        '--expected-official-commit', OFFICIAL_COMMIT,
        '--expected-checkpoint-sha256', EXPECTED_CHECKPOINT_SHA256,
    ])
    print('Runtime archive ready:', ARCHIVE)
else:
    print('Submission build is OFF.')


## Export only the final pretrained result

This is the notebook's only Drive write. It copies task3.zip and compact JSON
provenance/validation reports into the pretrained export area. It never copies
the full-volume prediction tree, caches, source data, or checkpoint.


In [ ]:
EXPORT_PRETRAINED_RESULT = False  # @param {type:'boolean'}
EXPORT_NAME_VALUE = 'official_pretrained_epoch50'  # @param {type:'string'}
if EXPORT_PRETRAINED_RESULT:
    if not ARCHIVE.is_file() or not SUBMISSION_REPORT.is_file():
        raise FileNotFoundError('Build and validate task3.zip before export.')
    export_dir = DRIVE_EXPORT_ROOT / EXPORT_NAME_VALUE
    if export_dir.exists():
        raise FileExistsError(
            f'Export already exists; choose a new name: {export_dir}'
        )
    export_dir.mkdir(parents=True)
    selected = [
        ARCHIVE,
        SUBMISSION_REPORT,
        REPORT_DIR / 'live_source_check.json',
        REPORT_DIR / 'pretrained_input_audit.json',
        REPORT_DIR / 'pretrained_smoke_validation.json',
        REPORT_DIR / 'pretrained_full_validation.json',
        FULL_OUTPUT / 'task3_run_manifest.json',
    ]
    copied = []
    for source in selected:
        if source.is_file():
            destination = export_dir / source.name
            shutil.copy2(source, destination)
            copied.append(destination)
    exported_archive = export_dir / ARCHIVE.name
    def sha256_path(path):
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    if sha256_path(exported_archive) != sha256_path(ARCHIVE):
        raise RuntimeError('Exported archive hash mismatch.')
    print('Saved final pretrained artifacts:')
    for path in copied:
        print(' -', path)
    print('Upload was not performed.')
else:
    print('Pretrained export is OFF; all results remain runtime-local.')
